# Financial Services Dashboard
Interactive KPI summary for Transaction Overview, Fraud & Risk, and Portfolio by Segment.

In [ ]:
df_daily    = spark.read.format('delta').table('gold_daily_transaction_summary').limit(100000).toPandas()
df_fraud    = spark.read.format('delta').table('gold_fraud_analysis').limit(100000).toPandas()
df_segment  = spark.read.format('delta').table('gold_segment_portfolio').limit(100000).toPandas()
df_risk     = spark.read.format('delta').table('gold_credit_risk_distribution').limit(100000).toPandas()
df_weekly   = spark.read.format('delta').table('gold_weekly_trends').limit(100000).toPandas()

In [ ]:
total_txns   = int(df_daily['transaction_count'].sum())
total_vol    = round(df_daily['total_amount'].sum(), 2)
total_fraud  = int(df_daily['fraud_count'].sum())
fraud_rate   = round(total_fraud / total_txns * 100, 3)
avg_txn      = round(df_daily['total_amount'].sum() / total_txns, 2)

print(f"Total Transactions : {total_txns:,}")
print(f"Total Volume       : £{total_vol:,.2f}")
print(f"Fraud Flagged      : {total_fraud:,} ({fraud_rate}%)")
print(f"Avg Transaction    : £{avg_txn:,.2f}")

In [ ]:
top_fraud = df_fraud.sort_values('fraud_rate', ascending=False).head(15)
seg_rows  = df_segment.sort_values('total_volume', ascending=False).head(12)

html = f"""
<!DOCTYPE html><html><head>
<meta charset="utf-8">
<title>Financial Services Dashboard</title>
<style>
  body  {{ font-family: Segoe UI, Arial, sans-serif; background:#f4f6f9; margin:0; padding:20px; }}
  h1    {{ color:#0078d4; border-bottom:3px solid #0078d4; padding-bottom:8px; }}
  h2    {{ color:#323130; margin-top:30px; }}
  .kpis {{ display:flex; gap:16px; flex-wrap:wrap; margin:20px 0; }}
  .kpi  {{ background:#fff; border-radius:8px; padding:20px 28px; box-shadow:0 2px 6px rgba(0,0,0,.08);
           min-width:160px; text-align:center; }}
  .kpi .val {{ font-size:2em; font-weight:700; color:#0078d4; }}
  .kpi .lbl {{ font-size:.85em; color:#605e5c; margin-top:4px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff; border-radius:8px;
           box-shadow:0 2px 6px rgba(0,0,0,.08); overflow:hidden; margin-bottom:24px; }}
  th    {{ background:#0078d4; color:#fff; padding:10px 14px; text-align:left; font-size:.9em; }}
  td    {{ padding:9px 14px; border-bottom:1px solid #edebe9; font-size:.88em; }}
  tr:hover td {{ background:#f3f9ff; }}
  .badge {{ display:inline-block; padding:2px 10px; border-radius:12px; font-size:.8em; font-weight:600; }}
  .Low    {{ background:#dff6dd; color:#107c10; }}
  .Medium {{ background:#cceaff; color:#0078d4; }}
  .High   {{ background:#fff4ce; color:#b7610e; }}
  .Critical {{ background:#fde7e9; color:#a80000; }}
</style></head><body>
<h1>💰 Financial Services Dashboard</h1>

<div class="kpis">
  <div class="kpi"><div class="val">{total_txns:,}</div><div class="lbl">Total Transactions</div></div>
  <div class="kpi"><div class="val">£{total_vol/1e6:.1f}M</div><div class="lbl">Total Volume</div></div>
  <div class="kpi"><div class="val">{fraud_rate}%</div><div class="lbl">Fraud Rate</div></div>
  <div class="kpi"><div class="val">£{avg_txn:,.0f}</div><div class="lbl">Avg Transaction</div></div>
</div>

<h2>Fraud Hotspots (top 15 by rate)</h2>
<table>
  <tr><th>Category</th><th>Country</th><th>Risk Band</th><th>Transactions</th><th>Fraud Count</th><th>Fraud Rate</th><th>Fraud Amount</th></tr>
  {''.join(f'''<tr>
    <td>{r["merchant_category"]}</td>
    <td>{r["country"]}</td>
    <td><span class="badge {r["risk_band"].replace(" ","")}">{r["risk_band"]}</span></td>
    <td>{int(r["total_transactions"]):,}</td>
    <td>{int(r["fraud_count"])}</td>
    <td>{r["fraud_rate"]}%</td>
    <td>£{r["fraud_amount"]:,.2f}</td>
  </tr>''' for _, r in top_fraud.iterrows())}
</table>

<h2>Portfolio by Segment</h2>
<table>
  <tr><th>Segment</th><th>Region</th><th>Risk Tier</th><th>Customers</th><th>Transactions</th><th>Volume</th><th>Avg Txn</th><th>Fraud Rate</th></tr>
  {''.join(f'''<tr>
    <td>{r["segment"]}</td>
    <td>{r["region"]}</td>
    <td><span class="badge {r["risk_tier"].replace(" ","")}">{r["risk_tier"]}</span></td>
    <td>{int(r["unique_customers"]):,}</td>
    <td>{int(r["total_transactions"]):,}</td>
    <td>£{r["total_volume"]/1e3:.1f}K</td>
    <td>£{r["avg_transaction_value"]:,.2f}</td>
    <td>{r["fraud_rate"]}%</td>
  </tr>''' for _, r in seg_rows.iterrows())}
</table>
</body></html>
"""

displayHTML(html)